<a href="https://colab.research.google.com/github/SattamAltwaim/EVE/blob/main/benchmarks/benchmark_eve_internals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EVE Internals: Offspring Selection, Fitness & Direction Analysis

**ResNet-18 on CIFAR-10 -- 80 epochs**

Diagnostic experiments that expose the internal dynamics of the EVE optimizer:

1. **Offspring weights & fitness** -- which offspring (d1-d4) dominate and when
2. **Direction divergence** -- cosine similarity between offspring directions
3. **Strength signal** -- per-dimension evolutionary memory dynamics
4. **EVE vs Adam + random perturbations** -- does structured offspring design beat noise?

### Setup
Run first cell to clone repo and install. Designed for **Colab A100 (CUDA)**, falls back to MPS or CPU.

In [ ]:
import os, sys, subprocess

BRANCH = "full-batch-probing"  # set to the desired branch or tag

if os.path.exists("/content"):
    repo_dir = "/content/EVE"
    if os.path.exists(repo_dir):
        # Pull latest instead of silently using a stale cached clone
        subprocess.run(["git", "-C", repo_dir, "fetch", "origin"], check=False)
        subprocess.run(["git", "-C", repo_dir, "checkout", BRANCH], check=False)
        subprocess.run(["git", "-C", repo_dir, "reset", "--hard", f"origin/{BRANCH}"], check=False)
        print(f"Updated existing clone to origin/{BRANCH}")
    else:
        subprocess.run(
            ["git", "clone", "--branch", BRANCH,
             "https://github.com/SattamAltwaim/EVE.git", repo_dir],
            check=False,
        )
        print(f"Cloned branch {BRANCH}")
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    # Print commit hash so you can verify which eve.py is loaded
    result = subprocess.run(
        ["git", "-C", repo_dir, "log", "--oneline", "-3"],
        capture_output=True, text=True,
    )
    print("Active commits:\n" + result.stdout.strip())
else:
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.exists(os.path.join(parent, "eve_optimizer")) and parent not in sys.path:
        sys.path.insert(0, parent)

# Force reimport in case a previous import is cached
import importlib, eve_optimizer
importlib.reload(eve_optimizer)
from eve_optimizer import EVE
print("EVE imported successfully")

In [ ]:
import math, time, copy
from collections import defaultdict
from typing import Any, Callable, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from torch.func import functional_call, vmap
from scipy.stats import beta as beta_dist
from sklearn.metrics import f1_score

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

plt.rcParams.update({
    "figure.dpi": 130, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9,
})
OFFSPRING_LABELS = ["d1 (Adam)", "d2 (Slow Momentum)", "d3 (Complementary)", "d4 (Virtual SAM)"]
OFFSPRING_COLORS = ["#2563eb", "#16a34a", "#ea580c", "#9333ea"]

In [ ]:
# ── Data: CIFAR-10 ────────────────────────────────────────────────────────
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])
transform_test = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])

full_train = torchvision.datasets.CIFAR10(root="./data", train=True,
                                          download=True, transform=transform_train)
test_set   = torchvision.datasets.CIFAR10(root="./data", train=False,
                                          download=True, transform=transform_test)

TRAIN_SUBSET = 5000
torch.manual_seed(42)
train_indices = torch.randperm(len(full_train))[:TRAIN_SUBSET].tolist()
train_set = Subset(full_train, train_indices)

BATCH_SIZE = 128
train_dl = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)
test_dl  = DataLoader(test_set, batch_size=512, shuffle=False,
                      num_workers=2, pin_memory=True)

print(f"Train subset: {len(train_set)}, Test: {len(test_set)}, "
      f"Batches/epoch: {len(train_dl)}, Batch size: {BATCH_SIZE}")

In [ ]:
# ── Model: ResNet-18 adapted for CIFAR-10 (32x32) ────────────────────────
def make_resnet18_cifar():
    """ResNet-18 with CIFAR-friendly stem (3x3 conv, no maxpool)."""
    model = torchvision.models.resnet18(num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model.to(DEVICE)

def make_model():
    return make_resnet18_cifar()

_tmp = make_model()
print(f"ResNet-18 (CIFAR) params: {sum(p.numel() for p in _tmp.parameters()):,}")
del _tmp

In [ ]:
# ── Evaluation helper ─────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, dataloader, loss_fn=None):
    """Return (accuracy, val_loss, macro_f1)."""
    model.eval()
    loss_fn = loss_fn or nn.CrossEntropyLoss()
    correct, total = 0, 0
    all_preds, all_labels = [], []
    total_loss = 0.0
    for xb, yb in dataloader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out = model(xb)
        loss = loss_fn(out, yb)
        total_loss += loss.item() * yb.size(0)
        pred = out.argmax(1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
        all_preds.extend(pred.cpu().numpy().tolist())
        all_labels.extend(yb.cpu().numpy().tolist())
    acc = correct / total
    val_loss = total_loss / total
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return acc, val_loss, macro_f1

In [ ]:
# ── Training loop for EVE with diagnostics ────────────────────────────────
EPOCHS = 50
RHO = 0.5

def train_eve_with_diagnostics(
    model, train_dl, test_dl, *,
    lr=1e-3, epochs=EPOCHS, K=4, rho=RHO,
    record_diagnostics=True,
):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = EVE(
        model.parameters(), lr=lr, K=K, rho=rho,
        record_diagnostics=record_diagnostics,
    )
    history = {
        "train_loss": [], "val_loss": [], "val_acc": [], "val_macro_f1": [],
        "step_losses": [], "batch_times": [],  # seconds per batch
    }

    for epoch in range(epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            t0 = time.perf_counter()
            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            optimizer.step(model=model, loss_fn=loss_fn, data=(xb, yb),
                           current_loss=loss.item())
            batch_time = time.perf_counter() - t0
            epoch_loss += loss.item()
            n_batches += 1
            history["step_losses"].append(loss.item())
            history["batch_times"].append(batch_time)

        history["train_loss"].append(epoch_loss / n_batches)
        acc, val_loss, macro_f1 = evaluate(model, test_dl, loss_fn=loss_fn)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(acc)
        history["val_macro_f1"].append(macro_f1)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            avg_bt = sum(history["batch_times"][-n_batches:]) / n_batches
            print(f"  Epoch {epoch+1:3d}/{epochs}  loss={history['train_loss'][-1]:.4f}  "
                  f"val_loss={val_loss:.4f}  val_acc={acc:.4f}  val_macro_f1={macro_f1:.4f}  "
                  f"beta_sel={optimizer.beta_sel:.3f}  avg_batch_time={avg_bt*1e3:.1f}ms")

    return optimizer, history

---
## Experiment 1: Offspring Fitness & Selection Weight Dynamics

Track how the 4 offspring compete over 80 epochs of ResNet-18 training on CIFAR-10.

Key questions:
- Does d1 (Adam) always dominate, or do alternatives win?
- When do d3 (complementary) and d4 (virtual SAM) gain weight?
- How does beta_sel (temperature) adapt?

In [ ]:
print("Training EVE (K=4) with diagnostics on ResNet-18 / CIFAR-10...")
torch.manual_seed(42)
model_diag = make_model()
opt_diag, hist_diag = train_eve_with_diagnostics(
    model_diag, train_dl, test_dl,
    lr=1e-3, epochs=EPOCHS, K=4, rho=RHO,
)
diag = opt_diag._diagnostics
print(f"\nCollected {len(diag)} diagnostic records over {EPOCHS} epochs.")
print(f"Final val accuracy: {hist_diag['val_acc'][-1]:.4f}  best val macro F1: {max(hist_diag['val_macro_f1']):.4f}")

In [ ]:
# ── Extract arrays ────────────────────────────────────────────────────────
steps     = np.array([d["step"] for d in diag])
w_arr     = np.array([d["weights"] for d in diag])       # (T, 4)
f_arr     = np.array([d["fitness"] for d in diag])       # (T, 4)
beta_arr  = np.array([d["beta_sel"] for d in diag])      # (T,)
loss_arr  = np.array([d["loss"] if d["loss"] is not None else np.nan for d in diag])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1a. Stacked area: selection weights
ax = axes[0, 0]
ax.stackplot(steps, w_arr.T, labels=OFFSPRING_LABELS, colors=OFFSPRING_COLORS, alpha=0.8)
ax.set_ylabel("Selection Weight (w_k)")
ax.set_xlabel("Step")
ax.set_title("Offspring Selection Weights Over Training")
ax.set_ylim(0, 1)
ax.legend(loc="upper right", fontsize=7)

# 1b. Fitness values
ax = axes[0, 1]
for k in range(4):
    ax.plot(steps, f_arr[:, k], label=OFFSPRING_LABELS[k],
            color=OFFSPRING_COLORS[k], alpha=0.6, linewidth=0.7)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)
ax.set_ylabel("Fitness (F_k = L_base - L_k)")
ax.set_xlabel("Step")
ax.set_title("Per-Offspring Fitness")
ax.legend(fontsize=7)

# 1c. beta_sel trajectory
ax = axes[1, 0]
ax.plot(steps, beta_arr, color="#dc2626", linewidth=1.5)
ax.set_ylabel(r"$\beta_{sel}$ (Selection Temperature)")
ax.set_xlabel("Step")
ax.set_title("Adaptive Selection Temperature")
ax.set_yscale("log")

# 1d. Training loss colored by dominant offspring
ax = axes[1, 1]
dominant = w_arr.argmax(axis=1)
valid_mask = ~np.isnan(loss_arr)
ax.plot(steps[valid_mask], loss_arr[valid_mask], color="black", linewidth=0.5, alpha=0.3)
for k in range(4):
    mask = (dominant == k) & valid_mask
    if mask.any():
        ax.scatter(steps[mask], loss_arr[mask], color=OFFSPRING_COLORS[k],
                   s=4, alpha=0.5, label=OFFSPRING_LABELS[k], zorder=3, edgecolors="none")
ax.set_ylabel("Training Loss")
ax.set_xlabel("Step")
ax.set_title("Loss Colored by Dominant Offspring")
ax.legend(fontsize=6, loc="upper right", markerscale=3)

fig.suptitle("Experiment 1: Offspring Selection Dynamics (K=4, ResNet-18, CIFAR-10)",
             fontsize=12, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# ── Log-scaled views for fine detail ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# 1e. Selection weights on log scale (reveals small contributions)
ax = axes[0]
for k in range(4):
    ax.plot(steps, w_arr[:, k], label=OFFSPRING_LABELS[k],
            color=OFFSPRING_COLORS[k], linewidth=0.8, alpha=0.7)
ax.set_yscale("log")
ax.set_ylabel("Selection Weight (log scale)")
ax.set_xlabel("Step")
ax.set_title("Selection Weights -- Log Scale")
ax.legend(fontsize=7)
ax.set_ylim(bottom=1e-6)

# 1f. Fitness on symlog scale (captures both positive and negative)
ax = axes[1]
for k in range(4):
    ax.plot(steps, f_arr[:, k], label=OFFSPRING_LABELS[k],
            color=OFFSPRING_COLORS[k], linewidth=0.8, alpha=0.7)
ax.set_yscale("symlog", linthresh=1e-4)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)
ax.set_ylabel("Fitness (symlog scale)")
ax.set_xlabel("Step")
ax.set_title("Per-Offspring Fitness -- Symlog Scale")
ax.legend(fontsize=7)

# 1g. Training loss on log scale
ax = axes[2]
valid = ~np.isnan(loss_arr)
ax.plot(steps[valid], loss_arr[valid], color="black", linewidth=0.8, alpha=0.7)
ax.set_yscale("log")
ax.set_ylabel("Training Loss (log scale)")
ax.set_xlabel("Step")
ax.set_title("Training Loss -- Log Scale")

fig.suptitle("Experiment 1 (cont.): Log-Scaled Views",
             fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
print("=" * 65)
print("Offspring Dominance Summary")
print("=" * 65)
dominant = w_arr.argmax(axis=1)
for k in range(4):
    pct = (dominant == k).mean() * 100
    avg_w = w_arr[:, k].mean()
    avg_f = f_arr[:, k].mean()
    print(f"  {OFFSPRING_LABELS[k]:25s}  dominant {pct:5.1f}%  "
          f"avg_weight={avg_w:.4f}  avg_fitness={avg_f:.6f}")

# Per-phase breakdown (early / mid / late)
T = len(diag)
phases = [("Early (0-33%)", 0, T//3), ("Mid (33-66%)", T//3, 2*T//3), ("Late (66-100%)", 2*T//3, T)]
print(f"\n{'Phase':<20s}", end="")
for lbl in OFFSPRING_LABELS:
    print(f"  {lbl:>15s}", end="")
print()
print("-" * 85)
for phase_name, i0, i1 in phases:
    print(f"{phase_name:<20s}", end="")
    for k in range(4):
        print(f"  {w_arr[i0:i1, k].mean():15.4f}", end="")
    print()

print(f"\nFinal beta_sel: {beta_arr[-1]:.4f}")
print(f"Final val accuracy: {hist_diag['val_acc'][-1]:.4f}  Best val accuracy:  {max(hist_diag['val_acc']):.4f}")
print(f"Best val macro F1: {max(hist_diag['val_macro_f1']):.4f}")

---
## Experiment 2: Offspring Direction Divergence

How different are the 4 offspring directions from each other? If they are nearly parallel, EVE gains little from selection. If they diverge, selection is choosing between genuinely different strategies.

On ResNet-18 we expect more divergence than on a small MLP since the parameter space mixes conv, BN, and FC layers with very different curvature profiles.

In [ ]:
pair_keys = list(diag[0]["cos_pairs"].keys())
cos_arr = np.array([[d["cos_pairs"][k] for k in pair_keys] for d in diag])
norm_arr = np.array([d["dir_norms"] for d in diag])
cos_comb_arr = np.array([d["cos_to_combined"] for d in diag])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# 2a. Pairwise cosine similarities
ax = axes[0]
pair_colors = ["#e63946", "#00b4d8", "#2a9d8f", "#e9c46a", "#9b5de5", "#ff6b00"]
for i, key in enumerate(pair_keys):
    ax.plot(steps, cos_arr[:, i], label=key, color=pair_colors[i % len(pair_colors)],
            linewidth=0.8, alpha=0.7)
ax.axhline(1.0, color="gray", linestyle=":", linewidth=0.5)
ax.axhline(0.0, color="gray", linestyle="--", linewidth=0.5)
ax.set_ylabel("Cosine Similarity")
ax.set_xlabel("Step")
ax.set_title("Pairwise Cosine Similarity Between Offspring")
ax.legend(fontsize=6, ncol=2)
ax.set_ylim(-0.3, 1.1)

# 2b. Direction norms
ax = axes[1]
for k in range(4):
    ax.plot(steps, norm_arr[:, k], label=OFFSPRING_LABELS[k],
            color=OFFSPRING_COLORS[k], linewidth=0.8, alpha=0.8)
ax.set_ylabel("L2 Norm")
ax.set_xlabel("Step")
ax.set_title("Offspring Direction Norms")
ax.legend(fontsize=7)

# 2c. Cosine to combined direction
ax = axes[2]
for k in range(4):
    ax.plot(steps, cos_comb_arr[:, k], label=OFFSPRING_LABELS[k],
            color=OFFSPRING_COLORS[k], linewidth=0.8, alpha=0.8)
ax.axhline(1.0, color="gray", linestyle=":", linewidth=0.5)
ax.set_ylabel("Cosine Similarity to Combined")
ax.set_xlabel("Step")
ax.set_title("How Much Does Each Offspring Influence the Update?")
ax.legend(fontsize=7)

fig.suptitle("Experiment 2: Offspring Direction Divergence (ResNet-18)",
             fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ── Log-scaled direction norms ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 2d. Direction norms on log scale
ax = axes[0]
for k in range(4):
    ax.plot(steps, norm_arr[:, k], label=OFFSPRING_LABELS[k],
            color=OFFSPRING_COLORS[k], linewidth=0.8, alpha=0.8)
ax.set_yscale("log")
ax.set_ylabel("L2 Norm (log scale)")
ax.set_xlabel("Step")
ax.set_title("Offspring Direction Norms -- Log Scale")
ax.legend(fontsize=7)

# 2e. Norm ratios relative to d1 (Adam)
ax = axes[1]
for k in range(1, 4):
    ratio = norm_arr[:, k] / np.clip(norm_arr[:, 0], 1e-12, None)
    ax.plot(steps, ratio, label=f"{OFFSPRING_LABELS[k]} / d1",
            color=OFFSPRING_COLORS[k], linewidth=0.8, alpha=0.8)
ax.axhline(1.0, color="gray", linestyle=":", linewidth=0.5)
ax.set_yscale("log")
ax.set_ylabel("Norm Ratio vs d1 (log scale)")
ax.set_xlabel("Step")
ax.set_title("Direction Magnitude Relative to Adam")
ax.legend(fontsize=7)

fig.suptitle("Experiment 2 (cont.): Log-Scaled Norms & Ratios",
             fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
print("=" * 65)
print("Direction Divergence Summary")
print("=" * 65)
for i, key in enumerate(pair_keys):
    avg_cos = cos_arr[:, i].mean()
    min_cos = cos_arr[:, i].min()
    print(f"  cos({key:8s})  avg={avg_cos:+.4f}  min={min_cos:+.4f}")
print()
for k in range(4):
    avg_norm = norm_arr[:, k].mean()
    avg_cos_c = cos_comb_arr[:, k].mean()
    print(f"  {OFFSPRING_LABELS[k]:25s}  avg_norm={avg_norm:.4f}  "
          f"avg_cos_to_combined={avg_cos_c:+.4f}")

---
## Experiment 3: Strength Signal Dynamics

The strength signal `s` is EVE's evolutionary memory. It tracks per-dimension historical success:
- `s -> 1` where Adam is succeeding
- `s -> 0` where Adam is failing

The complementary offspring d3 multiplies a momentum-gradient blend by `(1 - s)`, amplifying signal on low-s (failing) dimensions.

In [ ]:
s_records = [d for d in diag if d["s_stats"]]
s_steps = np.array([d["step"] for d in s_records])
s_mean = np.array([d["s_stats"]["mean"] for d in s_records])
s_std  = np.array([d["s_stats"]["std"]  for d in s_records])
s_min  = np.array([d["s_stats"]["min"]  for d in s_records])
s_max  = np.array([d["s_stats"]["max"]  for d in s_records])
s_med  = np.array([d["s_stats"]["median"] for d in s_records])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# 3a. Mean +/- std of strength signal
ax = axes[0]
ax.plot(s_steps, s_mean, color="#2563eb", linewidth=1.5, label="mean(s)")
ax.fill_between(s_steps, np.clip(s_mean - s_std, 0, 1),
                np.clip(s_mean + s_std, 0, 1),
                alpha=0.2, color="#2563eb", label=r"$\pm$ 1 std")
ax.plot(s_steps, s_med, color="#ea580c", linewidth=1, linestyle="--", label="median(s)")
ax.axhline(0.5, color="gray", linestyle=":", linewidth=0.5, label="neutral (0.5)")
ax.set_ylabel("Strength Signal (s)")
ax.set_xlabel("Step")
ax.set_title("Strength Signal Distribution Over Training")
ax.set_ylim(0, 1)
ax.legend(fontsize=7)

# 3b. Min / Max envelope
ax = axes[1]
ax.fill_between(s_steps, s_min, s_max, alpha=0.25, color="#9333ea", label="[min, max]")
ax.plot(s_steps, s_mean, color="#2563eb", linewidth=1.5, label="mean")
ax.plot(s_steps, s_med, color="#ea580c", linewidth=1, linestyle="--", label="median")
ax.axhline(0.5, color="gray", linestyle=":", linewidth=0.5)
ax.set_ylabel("Strength Signal (s)")
ax.set_xlabel("Step")
ax.set_title("Strength Signal Range")
ax.set_ylim(0, 1)
ax.legend(fontsize=7)

# 3c. Estimated s distribution at different training stages
ax = axes[2]
n_recs = len(s_records)
checkpoints = [0, n_recs//4, n_recs//2, 3*n_recs//4, n_recs-1]
stage_colors = ["#94a3b8", "#60a5fa", "#34d399", "#fbbf24", "#f87171"]
stage_labels = ["start", "25%", "50%", "75%", "end"]

x_range = np.linspace(0.001, 0.999, 300)
for idx, col, lbl in zip(checkpoints, stage_colors, stage_labels):
    rec = s_records[idx]
    mu = rec["s_stats"]["mean"]
    sigma = rec["s_stats"]["std"]
    sigma_sq = max(sigma**2, 1e-10)
    if mu * (1 - mu) > sigma_sq and 0.01 < mu < 0.99:
        alpha_p = max(mu * (mu * (1 - mu) / sigma_sq - 1), 0.2)
        beta_p = max((1 - mu) * (mu * (1 - mu) / sigma_sq - 1), 0.2)
        pdf = beta_dist.pdf(x_range, alpha_p, beta_p)
        ax.plot(x_range, pdf, color=col, linewidth=1.5,
                label=f"step {rec['step']} ({lbl})")
    else:
        ax.axvline(mu, color=col, linewidth=1.5, linestyle="--",
                   label=f"step {rec['step']} ({lbl}), mean={mu:.3f}")

ax.set_xlabel("Strength Signal (s)")
ax.set_ylabel("Density (Beta approx.)")
ax.set_title("Estimated s Distribution at Training Stages")
ax.legend(fontsize=6)

fig.suptitle("Experiment 3: Strength Signal -- Evolutionary Memory (ResNet-18)",
             fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ── Log-scaled std of strength signal ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 3d. Std(s) on log scale
ax = axes[0]
ax.plot(s_steps, s_std, color="#dc2626", linewidth=1)
ax.set_yscale("log")
ax.set_ylabel("Std(s) -- log scale")
ax.set_xlabel("Step")
ax.set_title("Strength Signal Spread (Std) Over Training")

# 3e. Deviation from neutral: |mean(s) - 0.5| on log scale
ax = axes[1]
dev = np.abs(s_mean - 0.5)
ax.plot(s_steps, dev, color="#7c3aed", linewidth=1)
ax.set_yscale("log")
ax.set_ylabel("|mean(s) - 0.5| -- log scale")
ax.set_xlabel("Step")
ax.set_title("Strength Signal Deviation from Neutral")

fig.suptitle("Experiment 3 (cont.): Log-Scaled Strength Signal Views",
             fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
print("=" * 65)
print("Strength Signal Summary")
print("=" * 65)
first_s = s_records[0]["s_stats"]
last_s  = s_records[-1]["s_stats"]
print(f"  Initial: mean={first_s['mean']:.4f}  std={first_s['std']:.4f}  "
      f"[{first_s['min']:.4f}, {first_s['max']:.4f}]")
print(f"  Final:   mean={last_s['mean']:.4f}  std={last_s['std']:.4f}  "
      f"[{last_s['min']:.4f}, {last_s['max']:.4f}]")
print()
if last_s["mean"] > 0.6:
    print("  -> s > 0.5 on average: Adam directions mostly succeeding.")
    print("     d3 (complementary) focuses on the minority of failing dimensions.")
elif last_s["mean"] < 0.4:
    print("  -> s < 0.5 on average: Adam is struggling on many dimensions.")
    print("     d3 (complementary) is broadly active with substantial correction.")
else:
    print("  -> s near 0.5: mixed signal -- some dimensions succeeding, some failing.")
    print("     d3 provides balanced complementary exploration.")

---
## Experiment 4: EVE vs AdamW

Does EVE's structured offspring design (Adam, Slow Momentum, Complementary, Virtual SAM) with **full-batch** probe-based fitness evaluation outperform plain AdamW?

Both optimizers are trained with the same hyperparameters. Per-batch wall-clock time is recorded to quantify EVE's computational overhead relative to the baseline.

In [ ]:
def train_generic(make_opt_fn, train_dl, test_dl, *,
                  epochs=EPOCHS, use_eve_api=False, label=""):
    """Generic training loop; returns (optimizer, history)."""
    torch.manual_seed(42)
    model = make_model()
    loss_fn = nn.CrossEntropyLoss()
    optimizer = make_opt_fn(model.parameters())
    hist = {
        "train_loss": [], "val_loss": [], "val_acc": [], "val_macro_f1": [],
        "step_losses": [], "batch_times": [],  # seconds per batch
    }

    for epoch in range(epochs):
        model.train()
        epoch_loss, n = 0.0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            t0 = time.perf_counter()
            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            if use_eve_api:
                optimizer.step(model=model, loss_fn=loss_fn, data=(xb, yb),
                               current_loss=loss.item())
            else:
                optimizer.step()
            batch_time = time.perf_counter() - t0
            epoch_loss += loss.item()
            n += 1
            hist["step_losses"].append(loss.item())
            hist["batch_times"].append(batch_time)
        hist["train_loss"].append(epoch_loss / n)
        acc, val_loss, macro_f1 = evaluate(model, test_dl, loss_fn=loss_fn)
        hist["val_loss"].append(val_loss)
        hist["val_acc"].append(acc)
        hist["val_macro_f1"].append(macro_f1)
        if (epoch + 1) % 20 == 0 or epoch == 0:
            avg_bt = sum(hist["batch_times"][-n:]) / n
            print(f"  [{label}] Epoch {epoch+1:3d}/{epochs}  "
                  f"loss={hist['train_loss'][-1]:.4f}  val_loss={val_loss:.4f}  "
                  f"val_acc={acc:.4f}  val_macro_f1={macro_f1:.4f}  "
                  f"avg_batch_time={avg_bt*1e3:.1f}ms")

    return optimizer, hist

In [ ]:
results = {}

print("=" * 50)
print("Training AdamW baseline...")
print("=" * 50)
_, results["AdamW"] = train_generic(
    lambda p: torch.optim.AdamW(p, lr=1e-3),
    train_dl, test_dl, epochs=EPOCHS, label="AdamW",
)

print()
print("=" * 50)
print(f"Training EVE (K=4, rho={RHO})...")
print("=" * 50)
opt_eve2, results["EVE"] = train_generic(
    lambda p: EVE(p, lr=1e-3, K=4, rho=RHO, record_diagnostics=True),
    train_dl, test_dl, epochs=EPOCHS, use_eve_api=True, label="EVE",
)

In [ ]:
opt_colors = {"AdamW": "#6b7280", "EVE": "#2563eb"}
OPTIMIZERS = ["AdamW", "EVE"]

fig, axes = plt.subplots(3, 3, figsize=(17, 12))

# ── Row 1: linear scale ────────────────────────────────────────────────────
# 4a. Training loss
ax = axes[0, 0]
for name in OPTIMIZERS:
    ax.plot(results[name]["train_loss"], label=name,
            color=opt_colors[name], linewidth=1.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("Training Loss")
ax.set_title("Training Loss"); ax.legend()

# 4b. Validation accuracy
ax = axes[0, 1]
for name in OPTIMIZERS:
    ax.plot(results[name]["val_acc"], label=name,
            color=opt_colors[name], linewidth=1.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Accuracy")
ax.set_title("Validation Accuracy"); ax.legend()

# 4c. EVE selection entropy
ax = axes[0, 2]
eve_diag2 = opt_eve2._diagnostics
eve_entropy = []
for d in eve_diag2:
    w = np.array(d["weights"])
    eve_entropy.append(-np.sum(w * np.log(np.clip(w, 1e-30, None))))
max_ent = np.log(4)
ax.plot(eve_entropy, label="EVE", color=opt_colors["EVE"], linewidth=0.6, alpha=0.6)
ax.axhline(max_ent, color="gray", linestyle=":", linewidth=0.5,
           label=f"H_max = ln(4) = {max_ent:.2f}")
ax.set_xlabel("Step"); ax.set_ylabel("Selection Entropy H(w)")
ax.set_title("EVE Selection Entropy"); ax.legend(fontsize=7)

# ── Row 2: val loss, val macro F1, training loss log ──────────────────────
# 4d. Validation loss
ax = axes[1, 0]
for name in OPTIMIZERS:
    ax.plot(results[name]["val_loss"], label=name,
            color=opt_colors[name], linewidth=1.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Loss")
ax.set_title("Validation Loss"); ax.legend()

# 4e. Validation macro F1
ax = axes[1, 1]
for name in OPTIMIZERS:
    ax.plot(results[name]["val_macro_f1"], label=name,
            color=opt_colors[name], linewidth=1.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("Val Macro F1")
ax.set_title("Validation Macro F1"); ax.legend()

# 4f. Training loss – log scale
ax = axes[1, 2]
for name in OPTIMIZERS:
    ax.plot(results[name]["train_loss"], label=name,
            color=opt_colors[name], linewidth=1.5)
ax.set_yscale("log"); ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss (log scale)")
ax.set_title("Training Loss – Log Scale"); ax.legend()

# ── Row 3: per-step smoothed loss, error rate, batch timing ───────────────
# 4g. Per-step loss – log scale (smoothed)
ax = axes[2, 0]
window = 20
for name in OPTIMIZERS:
    raw = np.array(results[name]["step_losses"])
    if len(raw) > window:
        smoothed = np.convolve(raw, np.ones(window) / window, mode="valid")
        ax.plot(smoothed, label=name, color=opt_colors[name],
                linewidth=0.8, alpha=0.8)
ax.set_yscale("log"); ax.set_xlabel("Step")
ax.set_ylabel("Step Loss (log, smoothed)")
ax.set_title(f"Per-Step Loss – Log Scale (smoothed, window={window})"); ax.legend(fontsize=7)

# 4h. Validation error rate – log scale
ax = axes[2, 1]
for name in OPTIMIZERS:
    err = 1.0 - np.array(results[name]["val_acc"])
    ax.plot(err, label=name, color=opt_colors[name], linewidth=1.5)
ax.set_yscale("log"); ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Error Rate (log scale)")
ax.set_title("Validation Error Rate – Log Scale"); ax.legend()

# 4i. Time per batch – smoothed
ax = axes[2, 2]
window_t = 50
for name in OPTIMIZERS:
    raw_t = np.array(results[name]["batch_times"]) * 1e3  # ms
    if len(raw_t) > window_t:
        smoothed_t = np.convolve(raw_t, np.ones(window_t) / window_t, mode="valid")
        ax.plot(smoothed_t, label=name, color=opt_colors[name],
                linewidth=0.8, alpha=0.8)
ax.set_xlabel("Step"); ax.set_ylabel("Batch Time (ms)")
ax.set_title(f"Time per Batch (smoothed, window={window_t})"); ax.legend(fontsize=7)

fig.suptitle("Experiment 4: EVE vs AdamW (ResNet-18, CIFAR-10, full-batch probe)",
             fontsize=12, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
print("=" * 85)
print("Final Results: EVE vs AdamW  (full-batch probe)")
print("=" * 85)
print(f"{'Optimizer':15s} {'Best Val F1':>12s} {'Final Val F1':>12s} {'Best Val Acc':>12s} "
      f"{'Final Val Acc':>13s} {'Final Loss':>10s} {'Best Epoch':>10s} "
      f"{'Avg Batch ms':>13s}")
print("-" * 85)
for name in OPTIMIZERS:
    best_f1   = max(results[name]["val_macro_f1"])
    final_f1  = results[name]["val_macro_f1"][-1]
    best_ep   = np.argmax(results[name]["val_macro_f1"]) + 1
    best_acc  = max(results[name]["val_acc"])
    final_acc = results[name]["val_acc"][-1]
    final_loss = results[name]["train_loss"][-1]
    avg_bt_ms  = np.mean(results[name]["batch_times"]) * 1e3
    print(f"{name:15s} {best_f1:12.4f} {final_f1:12.4f} {best_acc:12.4f} "
          f"{final_acc:13.4f} {final_loss:10.4f} {best_ep:10d} {avg_bt_ms:13.1f}")

print()
eve_best  = max(results["EVE"]["val_macro_f1"])
adam_best = max(results["AdamW"]["val_macro_f1"])
eve_avg_t  = np.mean(results["EVE"]["batch_times"]) * 1e3
adam_avg_t = np.mean(results["AdamW"]["batch_times"]) * 1e3

print("Key Takeaways (by val macro F1):")
if eve_best > adam_best:
    print(f"  [1] EVE improves over AdamW (macro F1: {eve_best:.4f} vs {adam_best:.4f})")
else:
    print(f"  [1] AdamW matched/beat EVE (macro F1: {adam_best:.4f} vs {eve_best:.4f})")
print(f"  [2] Avg batch time — AdamW: {adam_avg_t:.1f}ms  EVE: {eve_avg_t:.1f}ms  "
      f"overhead: {(eve_avg_t/adam_avg_t - 1)*100:.1f}%")

In [ ]:
# Timing summary across all epochs
print("Batch-time summary (ms):")
print(f"  {'Optimizer':10s}  {'Mean':>8s}  {'Median':>8s}  {'p95':>8s}  {'Max':>8s}")
for name in OPTIMIZERS:
    bt = np.array(results[name]["batch_times"]) * 1e3
    print(f"  {name:10s}  {bt.mean():8.2f}  {np.median(bt):8.2f}  "
          f"{np.percentile(bt, 95):8.2f}  {bt.max():8.2f}")

---
## Experiment 5: EVE Phase-by-Phase Timing Diagnostic (Post-fix)

Instruments each internal phase of the **fixed** `EVE.step()` implementation with `torch.cuda.synchronize()` to get accurate GPU wall-clock times per phase.

The fixed implementation uses **in-place perturbation** for the probe (no `functional_call`, no `stacked_params`, no `vmap`) and passes `current_loss` from the caller (no hidden extra forward pass).

Phases timed:
- **P_str** — strength signal update
- **P2a** — moment updates + **single** `.item()` for `global_max_sqrt_v` (fixed: was 62 serial syncs)
- **P2b** — offspring direction construction (`torch.stack` per param)
- **P_save** — parameter backup clone (1 × D floats, ~44 MB for ResNet-18)
- **P4a** — `L_base` forward pass at θ_t (eval mode)
- **P4b** — K in-place probe cycles: perturb → `model()` forward → restore (no `functional_call`)
- **P5** — fitness + softmax
- **P6** — weighted `einsum` update
- **P7** — adaptive temperature

Eliminated vs old diagnostic:
- ~~P0~~ extra `current_loss` forward pass — caller now passes `current_loss=loss.item()`
- ~~P3~~ `stacked_params` dict build — replaced by in-place perturbation

In [ ]:
import math as _math

def _cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def diagnose_eve_phases_fixed(loss_fn, train_dl, K=4, n_warmup=5, n_measure=20):
    """
    Instruments the FIXED EVE.step() (in-place perturbation probe) with
    torch.cuda.synchronize() before each time.perf_counter() call.
    Eliminated vs old: P0 hidden fwd, P3 stacked_params, vmap/functional_call.
    Returns dict: phase_name -> list[float] of per-step ms values.
    """
    torch.manual_seed(0)
    model = make_model()
    opt = EVE(model.parameters(), lr=1e-3, K=K, rho=0.5, record_diagnostics=False)

    phase_names = [
        "P_str__strength_signal",
        "P2a_moments_global_max",
        "P2b_offspring_construction",
        "P_save_param_backup",
        "P4a_L_base_fwd",
        "P4b_K_inplace_probes",
        "P5_fitness_softmax",
        "P6_weighted_einsum_update",
        "P7_temperature_adapt",
    ]
    phase_times = {p: [] for p in phase_names}

    data_iter = iter(train_dl)

    for step_idx in range(n_warmup + n_measure):
        try:
            xb, yb = next(data_iter)
        except StopIteration:
            data_iter = iter(train_dl)
            xb, yb = next(data_iter)
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        # Standard forward + backward
        model.train()
        opt.zero_grad()
        out = model(xb)
        loss_val = loss_fn(out, yb)
        loss_val.backward()
        current_loss_val = loss_val.item()  # passed to step (Fix Bug 2)

        measuring = step_idx >= n_warmup
        if not measuring:
            opt.step(model=model, loss_fn=loss_fn, data=(xb, yb),
                     current_loss=current_loss_val)
            continue

        # ── P_str: strength signal update ──────────────────────────
        _cuda_sync(); t0 = time.perf_counter()
        opt._update_strength_signal(current_loss_val)
        _cuda_sync(); t_str = time.perf_counter()
        phase_times["P_str__strength_signal"].append((t_str - t0) * 1e3)

        # ── P2a: moments + SINGLE .item() for global_max_sqrt_v ────
        sqrt_v_hat_map_d = {}
        params_with_grad_d = []
        for group in opt.param_groups:
            beta1, beta2 = group["betas"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = opt.state[p]
                if len(state) == 0:
                    state["step"] = 0
                    state["m"] = torch.zeros_like(p)
                    state["v"] = torch.zeros_like(p)
                    state["s"] = torch.full_like(p, 0.5)
                    state["prev_update_sign"] = torch.zeros_like(p)
                state["step"] += 1
                m, v = state["m"], state["v"]
                m.mul_(beta1).add_(p.grad, alpha=1.0 - beta1)
                v.mul_(beta2).addcmul_(p.grad, p.grad, value=1.0 - beta2)
                bc2 = 1.0 - beta2 ** state["step"]
                sv = (v / bc2).sqrt()
                sqrt_v_hat_map_d[p.data_ptr()] = sv
                params_with_grad_d.append((group, p))
        # Single GPU reduction — one .item() for all params (Fix Bug 1)
        global_max_sqrt_v = torch.stack(
            [sv.max() for sv in sqrt_v_hat_map_d.values()]
        ).max().item()
        _cuda_sync(); t2a = time.perf_counter()
        phase_times["P2a_moments_global_max"].append((t2a - t_str) * 1e3)

        # ── P2b: offspring direction construction ──────────────────
        offspring_map_d = {}
        for group, p in params_with_grad_d:
            grad = p.grad
            state = opt.state[p]
            eps = group["eps"]
            grp_K = group["K"]
            beta1 = group["betas"][0]
            m, s = state["m"], state["s"]
            bc1 = 1.0 - beta1 ** state["step"]
            m_hat = m / bc1
            sv = sqrt_v_hat_map_d[p.data_ptr()]
            denom = sv + eps
            dirs = [m_hat.neg() / denom]
            if grp_K >= 2:
                dirs.append(grad.neg() / denom)
            if grp_K >= 3:
                dirs.append((grad.neg() * (1.0 - s)) / denom)
            if grp_K >= 4:
                dirs.append(grad.sign().neg() * (sv / (global_max_sqrt_v + eps)))
            offspring_map_d[p.data_ptr()] = torch.stack(dirs)
        _cuda_sync(); t2b = time.perf_counter()
        phase_times["P2b_offspring_construction"].append((t2b - t2a) * 1e3)

        # ── P_save: parameter backup clone ─────────────────────────
        saved_d = {p.data_ptr(): p.data.clone() for _, p in params_with_grad_d}
        _cuda_sync(); t_save = time.perf_counter()
        phase_times["P_save_param_backup"].append((t_save - t2b) * 1e3)

        # ── P4a: L_base forward pass at θ_t (eval mode) ────────────
        was_training = model.training
        model.eval()
        with torch.no_grad():
            L_base = loss_fn(model(xb), yb)
        _cuda_sync(); t4a = time.perf_counter()
        phase_times["P4a_L_base_fwd"].append((t4a - t_save) * 1e3)

        # ── P4b: K in-place probe cycles ────────────────────────────
        probe_losses = []
        with torch.no_grad():
            for k in range(K):
                for group, p in params_with_grad_d:
                    p.data.add_(offspring_map_d[p.data_ptr()][k], alpha=group["lr"])
                probe_losses.append(loss_fn(model(xb), yb))
                for _, p in params_with_grad_d:
                    p.data.copy_(saved_d[p.data_ptr()])
        losses = torch.stack(probe_losses)
        model.train(was_training)
        _cuda_sync(); t4b = time.perf_counter()
        phase_times["P4b_K_inplace_probes"].append((t4b - t4a) * 1e3)

        # ── P5: fitness + softmax ───────────────────────────────────
        fitness = L_base - losses
        weights = torch.softmax(opt.beta_sel * fitness, dim=0)
        _cuda_sync(); t5 = time.perf_counter()
        phase_times["P5_fitness_softmax"].append((t5 - t4b) * 1e3)

        # ── P6: weighted einsum update ──────────────────────────────
        for group, p in params_with_grad_d:
            dir_stack = offspring_map_d[p.data_ptr()]
            lr, wd = group["lr"], group["weight_decay"]
            state = opt.state[p]
            combined = torch.einsum("k...,k->...", dir_stack, weights)
            with torch.no_grad():
                state["prev_update_sign"] = (
                    (combined - wd * p.data).sign() if wd != 0.0 else combined.sign()
                )
                if wd != 0.0:
                    p.data.mul_(1.0 - lr * wd)
                p.data.add_(combined, alpha=lr)
        _cuda_sync(); t6 = time.perf_counter()
        phase_times["P6_weighted_einsum_update"].append((t6 - t5) * 1e3)

        # ── P7: adaptive temperature ────────────────────────────────
        rho = opt.defaults["rho"]
        alpha_beta = opt.defaults["alpha_beta"]
        beta_min, beta_max = opt.defaults["beta_sel_range"]
        H_t = -(weights * torch.log(weights.clamp(min=1e-30))).sum().item()
        opt.beta_sel *= _math.exp(alpha_beta * (H_t - rho * _math.log(K)))
        opt.beta_sel = max(beta_min, min(beta_max, opt.beta_sel))
        opt._prev_loss = current_loss_val
        _cuda_sync(); t7 = time.perf_counter()
        phase_times["P7_temperature_adapt"].append((t7 - t6) * 1e3)

    return phase_times


print("Running POST-FIX EVE phase-by-phase timing diagnostic...")
print(f"Device: {DEVICE}  |  K=4  |  5 warmup + 20 measured steps")
print("(in-place perturbation, no functional_call, no stacked_params)\n")

phase_times_fixed = diagnose_eve_phases_fixed(
    nn.CrossEntropyLoss(), train_dl, K=4, n_warmup=5, n_measure=20
)

phase_labels_fixed = {
    "P_str__strength_signal":     "P_str strength signal update",
    "P2a_moments_global_max":      "P2a  moments + single .item()      [FIXED: was 62 syncs]",
    "P2b_offspring_construction":  "P2b  offspring construction         (torch.stack per param)",
    "P_save_param_backup":         "P_sv param backup clone             (1x44MB vs old 4x176MB)",
    "P4a_L_base_fwd":              "P4a  L_base forward pass            (eval mode, theta_t)",
    "P4b_K_inplace_probes":        "P4b  K in-place probe cycles        [FIXED: was functional_call]",
    "P5_fitness_softmax":          "P5   fitness + softmax",
    "P6_weighted_einsum_update":   "P6   weighted einsum update",
    "P7_temperature_adapt":        "P7   temperature adaptation",
}

total_mean_fixed = sum(np.mean(v) for v in phase_times_fixed.values())

print(f"{'Phase':<60s} {'Mean ms':>8s} {'p50 ms':>8s} {'p95 ms':>8s} {'% total':>8s}")
print("\u2500" * 96)
for key, label in phase_labels_fixed.items():
    vals = np.array(phase_times_fixed[key])
    print(f"  {label:<58s} {vals.mean():8.2f} {np.median(vals):8.2f} "
          f"{np.percentile(vals,95):8.2f} {100*vals.mean()/total_mean_fixed:7.1f}%")
print("\u2500" * 96)
print(f"  {'TOTAL fixed EVE phases (excl. base training fwd+bwd)':<58s} {total_mean_fixed:8.2f}")

print()
print("Eliminated vs OLD diagnostic (shown for reference):")
print("  P0  extra current_loss fwd  old ~10.6ms  -> 0.0 ms  [caller passes current_loss]",)
print("  P3  stacked_params build     old  ~4.4ms  -> 0.0 ms  [in-place perturb replaces this]")
print("  P4b functional_call probes   old ~112ms   -> see P4b  [direct model() fwd per offspring]")
print()
OLD_TOTAL = 172.12
print(f"  Old instrumented total:  {OLD_TOTAL:.1f} ms")
print(f"  New instrumented total:  {total_mean_fixed:.1f} ms")
if total_mean_fixed > 0:
    print(f"  Instrumented speedup:    {OLD_TOTAL/total_mean_fixed:.1f}x")
print()
print("Expected real-world batch time: base fwd+bwd (~11ms) + EVE phases above")
print("Run Experiment 4 to measure actual batch times end-to-end.")


---
## Summary

These experiments reveal the internal dynamics of EVE on a realistic vision task (ResNet-18, CIFAR-10, 50 epochs) using **full-batch probing** (probe batch = full training batch):

1. **Selection weights** shift during training — offspring dominance is not static. The adaptive temperature `beta_sel` modulates how decisively EVE picks winners.

2. **Direction divergence** is genuine — especially d4 (virtual SAM) which probes sharpness in the neighborhood of the Adam step. The offspring are not redundant perturbations of Adam.

3. **Strength signal** differentiates successful vs failing dimensions, providing evolutionary memory that guides the complementary offspring d3 toward the dimensions that need it most.

4. **EVE vs AdamW** — Experiment 4 measures whether EVE's structured offspring selection (Adam, Slow Momentum, Complementary, Virtual SAM) with full-batch fitness evaluation outperforms plain AdamW, and quantifies the wall-clock overhead via per-batch timing.